# 05 · Statistical validation

Answers Reviewer 1 point 6.

### Why not chi-square testing on event distributions

With tens of millions of events, conventional chi-square tests on activity
counts are dominated by sample size. The inferential analysis therefore focuses
on paired model-quality comparisons across faculty-role partitions, together
with effect sizes and trace-level bootstrap uncertainty.

### What is done

* **Algorithm comparison.** Up to 14 paired observations (7 faculties × 2 roles)
  within each case notion. Friedman across the three miners per dimension,
  followed by Wilcoxon signed-rank pairwise comparisons with Holm correction.
  Effect sizes: Kendall's *W* and matched-pairs rank-biserial correlation.
* **Uncertainty.** Trace-level bootstrap sampling **with replacement**. Repeated
  draws of the same original trace are retained as distinct bootstrap cases.
  Percentile confidence intervals are computed for fitness, precision,
  generalization and simplicity.

Run notebook **02** first; this notebook reads `/content/02_quality.csv`.


## 1. Setup

Run this section first. It installs dependencies and downloads the deposit from
figshare into the Colab VM.

**Runtime:** Runtime &rarr; Change runtime type &rarr; **High-RAM** if available.
The largest faculty file (FIF, 1.3 GB on disk) needs roughly 6 GB once loaded.

In [ ]:
#@title Install dependencies { display-mode: "form" }
!pip install -q pm4py==2.7.23.3 statsmodels 2>/dev/null
import os
os.environ["TQDM_DISABLE"] = "1"

import warnings, sys, json, time, gc, random
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np

print("Python ", sys.version.split()[0])
print("pandas ", pd.__version__)
import pm4py; print("pm4py  ", pm4py.__version__)

# --- RAM report -------------------------------------------------------------
try:
    import psutil
    gb = psutil.virtual_memory().total / 1e9
    print(f"RAM    {gb:.1f} GB")
    if gb < 20:
        print("\\n  NOTE: standard runtime. FIF and FTE may run out of memory.")
        print("  Runtime -> Change runtime type -> High-RAM is recommended.")
except Exception:
    pass

In [ ]:
#@title Download the deposit from figshare { display-mode: "form" }
# Queries the figshare API, so lecturer files are picked up automatically
# once they are added to the deposit.

import requests, os, pathlib

ARTICLE = "28341992"          #@param {type:"string"}
DATA_DIR = "/content/data"    #@param {type:"string"}
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

meta = requests.get(f"https://api.figshare.com/v2/articles/{ARTICLE}", timeout=60).json()
print(f"{meta['title']}  (v{meta.get('version','?')})")
print(f"{len(meta['files'])} files, {meta['size']/1e9:.2f} GB total\n")

FILES = {}
for f in meta["files"]:
    FILES[f["name"]] = f["download_url"]
    print(f"  {f['name']:<32} {f['size']/1e6:>8.1f} MB")

# --- completeness check -----------------------------------------------------
FACULTIES = ["FEB", "FIF", "FIK", "FIT", "FKB", "FRI", "FTE"]
missing = [f"{fac}_{role}.csv" for fac in FACULTIES
           for role in ("Student", "Lecturer") if f"{fac}_{role}.csv" not in FILES]
if missing:
    print("\n  MISSING FROM DEPOSIT:")
    for m in missing:
        print(f"    {m}")
    print("\n  Analyses for these partitions will be skipped.")


def fetch(name):
    """Download one file if not already present. Returns local path or None."""
    if name not in FILES:
        return None
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        return dest
    print(f"downloading {name} ...", flush=True)
    with requests.get(FILES[name], stream=True, timeout=1800) as r:
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 22):
                fh.write(chunk)
    print(f"  -> {os.path.getsize(dest)/1e6:.0f} MB")
    return dest

In [ ]:
#@title Loader { display-mode: "form" }
# Memory-efficient reader plus the two corrections identified during revision.

USECOLS = ["id", "eventname", "component", "action", "target", "crud",
           "edulevel", "userid", "courseid", "timecreated", "event"]
DTYPES = {"id": "int64", "eventname": "category", "component": "category",
          "action": "category", "target": "category", "crud": "category",
          "edulevel": "int8", "userid": "int32", "courseid": "int32",
          "event": "category"}

CUTOFF = "2023-06-26"   # verified coverage boundary (NOT July, see manuscript)


def load(faculty, role, apply_dedup=True, cols=None):
    """Load one faculty-role partition.

    apply_dedup fixes the defect found during revision: the original notebooks
    called df.drop_duplicates() WITHOUT assignment, so duplicates were counted
    and reported but never removed from the working data.
    """
    name = f"{faculty}_{role}.csv"
    path = fetch(name)
    if path is None:
        print(f"  [skip] {name} not in deposit")
        return None
    use = cols or USECOLS
    df = pd.read_csv(path, index_col=0, usecols=lambda c: c in use or c == "Unnamed: 0",
                     dtype={k: v for k, v in DTYPES.items() if k in use},
                     parse_dates=["timecreated"] if "timecreated" in use else None)
    n_raw = len(df)
    n_dup = int(df.duplicated().sum())
    if apply_dedup and n_dup:
        df = df.drop_duplicates()          # assignment: this is the fix
    df.attrs["n_raw"] = n_raw
    df.attrs["n_dup"] = n_dup
    df.attrs["faculty"] = faculty
    df.attrs["role"] = role
    return df


def add_case(df, notion):
    """notion is 'user' or 'course'."""
    if notion == "user":
        df["case"] = df["userid"].astype(str)
    else:
        df["case"] = df["userid"].astype(str) + "_" + df["courseid"].astype(str)
    return df


def to_log(df, notion, sample=None, seed=42, max_len=None):
    """Build a pm4py EventLog. sample caps the number of traces."""
    d = add_case(df, notion)
    if max_len:
        keep = d.groupby("case").size()
        d = d[d["case"].isin(keep[keep <= max_len].index)]
    if sample:
        random.seed(seed)
        cases = sorted(d["case"].unique())
        d = d[d["case"].isin(set(random.sample(cases, min(sample, len(cases)))))]
    ldf = (d[["case", "event", "timecreated"]]
           .rename(columns={"case": "case:concept:name", "event": "concept:name",
                            "timecreated": "time:timestamp"})
           .sort_values(["case:concept:name", "time:timestamp"])
           .reset_index(drop=True))
    # pm4py rejects categorical columns: cast the two key columns to str
    ldf["case:concept:name"] = ldf["case:concept:name"].astype(str)
    ldf["concept:name"] = ldf["concept:name"].astype(str)
    return pm4py.convert_to_event_log(ldf), ldf


def save(obj, name):
    """Persist a result table and offer it for download."""
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(f"/content/{name}.csv", index=False)
    else:
        json.dump(obj, open(f"/content/{name}.json", "w"), indent=1)
    print(f"saved /content/{name}")

## 2. Friedman and Wilcoxon on the algorithm comparison

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from statsmodels.stats.multitest import multipletests

QUALITY_FILE = "/content/02_quality.csv"
if not os.path.exists(QUALITY_FILE):
    raise FileNotFoundError(
        "02_quality.csv not found. Run 02_model_quality.ipynb first in the same runtime, "
        "or upload its output to /content/02_quality.csv.")
quality = pd.read_csv(QUALITY_FILE)
q = quality[quality["error"].isna()] if "error" in quality.columns else quality

stat_rows = []

for notion in q.notion.unique():
    sub = q[q.notion == notion]
    print(f"\n{'='*64}\n{notion}-level case notion\n{'='*64}")
    for dim in ["fitness", "precision", "generalization", "simplicity"]:
        piv = sub.pivot_table(
            index=["faculty", "role"], columns="miner", values=dim).dropna()
        required = {"alpha", "heuristic", "inductive"}
        if len(piv) < 3 or not required.issubset(piv.columns):
            print(f"\n{dim}: only {len(piv)} complete pairs, skipped")
            continue
        piv = piv[["alpha", "heuristic", "inductive"]]
        a, h, i = piv["alpha"], piv["heuristic"], piv["inductive"]
        stat, p = friedmanchisquare(a, h, i)
        k, n = 3, len(piv)
        W = stat / (n * (k - 1))          # Kendall's W
        stat_rows.append(dict(
            notion=notion, dimension=dim, test="Friedman",
            comparison="alpha vs heuristic vs inductive",
            n=n, statistic=stat, p_raw=p, p_holm=np.nan,
            effect_name="Kendall_W", effect=W))
        print(f"\n{dim}  (n={n} faculty-role pairs)")
        print(f"  Friedman chi2={stat:.3f}, p={p:.2e}, Kendall W={W:.3f}")
        print("  means: " + ", ".join(
            f"{m}={piv[m].mean():.4f}" for m in piv.columns))

        pair_defs = [("alpha", "heuristic"), ("alpha", "inductive"),
                     ("heuristic", "inductive")]
        pair_tmp = []
        for x, y in pair_defs:
            try:
                s, pv = wilcoxon(piv[x], piv[y])
                d = piv[x] - piv[y]
                d_nz = d[d != 0]
                if len(d_nz):
                    signed_ranks = rankdata(abs(d_nz)) * np.sign(d_nz)
                    rbc = signed_ranks.sum() / (
                        len(d_nz) * (len(d_nz) + 1) / 2)
                else:
                    rbc = 0.0
                pair_tmp.append([x, y, s, pv, rbc])
            except Exception:
                pair_tmp.append([x, y, np.nan, np.nan, np.nan])

        raw_ps = np.array([r[3] for r in pair_tmp], dtype=float)
        ok = ~np.isnan(raw_ps)
        adj = np.full(len(raw_ps), np.nan)
        if ok.any():
            adj[ok] = multipletests(raw_ps[ok], method="holm")[1]

        for (x, y, s, pv, rbc), av in zip(pair_tmp, adj):
            stat_rows.append(dict(
                notion=notion, dimension=dim, test="Wilcoxon signed-rank",
                comparison=f"{x} vs {y}", n=n, statistic=s,
                p_raw=pv, p_holm=av,
                effect_name="rank_biserial", effect=rbc))
            print(f"    {x+' vs '+y:<24} p={pv:.4g}  "
                  f"p_holm={av:.4g}  rank-biserial={rbc:+.3f}")

stats = pd.DataFrame(stat_rows)
save(stats, "05_algorithm_stats")
stats


## 3. Bootstrap confidence intervals

Resamples **traces with replacement** and recomputes the process model and all
four quality metrics. If one original case is drawn multiple times, each draw
is copied with a new bootstrap case identifier so that multiplicity is
preserved. This corrects the common but invalid pattern of converting sampled
case IDs to a `set`, which silently removes duplicate bootstrap draws.

The default uses 100 replicates as a practical compromise for reviewer
reproduction. Increase `N_BOOT` (for example, to 500 or 1000) if compute
resources permit and report the final value used.


In [ ]:
#@title { display-mode: "form" }
BOOT_FACULTY = "FIT"      #@param ["FEB","FIF","FIK","FIT","FKB","FRI","FTE"]
BOOT_ROLE    = "Student"  #@param ["Student","Lecturer"]
BOOT_NOTION  = "course"   #@param ["course","user"]
N_BOOT       = 100        #@param {type:"integer"}
BOOT_SAMPLE  = 120        #@param {type:"integer"}
BOOT_MAXLEN  = 2000       #@param {type:"integer"}
BOOT_SEED    = 1000       #@param {type:"integer"}

from pm4py.algo.discovery.alpha import algorithm as alpha_miner
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.algo.evaluation.replay_fitness import algorithm as fit_eval
from pm4py.algo.evaluation.precision import algorithm as prec_eval
from pm4py.algo.evaluation.generalization import algorithm as gen_eval
from pm4py.algo.evaluation.simplicity import algorithm as sim_eval
P = {"show_progress_bar": False}

df = load(BOOT_FACULTY, BOOT_ROLE)
df = add_case(df, BOOT_NOTION)

# Apply the same trace-length guard used in the model-quality notebook.
case_len = df.groupby("case").size()
eligible = sorted(case_len[case_len <= BOOT_MAXLEN].index.astype(str))
df["case"] = df["case"].astype(str)
df = df[df["case"].isin(eligible)].copy()

if not eligible:
    raise ValueError("No eligible traces remain after BOOT_MAXLEN filtering.")

boot = {m: {"fitness": [], "precision": [], "generalization": [], "simplicity": []}
        for m in ["alpha", "heuristic", "inductive"]}


def bootstrap_log(source_df, sampled_cases):
    """Copy sampled traces with multiplicity preserved.

    Each draw gets a unique bootstrap case ID. Thus, if an original trace is
    sampled three times, it appears three times in the bootstrap event log.
    """
    pieces = []
    grouped = {cid: g[["event", "timecreated"]].copy()
               for cid, g in source_df.groupby("case", sort=False)}
    for draw_idx, cid in enumerate(sampled_cases):
        g = grouped[str(cid)].copy()
        g["case:concept:name"] = f"boot_{draw_idx:05d}::{cid}"
        pieces.append(g)
    d = pd.concat(pieces, ignore_index=True)
    d = (d.rename(columns={"event": "concept:name",
                           "timecreated": "time:timestamp"})
          [["case:concept:name", "concept:name", "time:timestamp"]]
          .sort_values(["case:concept:name", "time:timestamp"])
          .reset_index(drop=True))
    d["case:concept:name"] = d["case:concept:name"].astype(str)
    d["concept:name"] = d["concept:name"].astype(str)
    return pm4py.convert_to_event_log(d), d


for b in range(N_BOOT):
    rng = np.random.default_rng(BOOT_SEED + b)
    sampled = rng.choice(
        eligible, size=min(BOOT_SAMPLE, len(eligible)), replace=True)
    lg, ldf = bootstrap_log(df, sampled)

    for m in boot:
        try:
            if m == "alpha":
                net, im, fm = alpha_miner.apply(lg)
            elif m == "heuristic":
                net, im, fm = heuristics_miner.apply(
                    lg, variant=heuristics_miner.Variants.CLASSIC)
            else:
                t = inductive_miner.apply(
                    lg, variant=inductive_miner.Variants.IM)
                net, im, fm = (pm4py.convert_to_petri_net(t)
                               if not isinstance(t, tuple) else t)

            boot[m]["fitness"].append(fit_eval.apply(
                lg, net, im, fm,
                variant=fit_eval.Variants.TOKEN_BASED,
                parameters=P)["log_fitness"])
            boot[m]["precision"].append(prec_eval.apply(
                lg, net, im, fm,
                variant=prec_eval.Variants.ETCONFORMANCE_TOKEN,
                parameters=P))
            boot[m]["generalization"].append(
                gen_eval.apply(lg, net, im, fm))
            boot[m]["simplicity"].append(sim_eval.apply(net))
        except Exception as e:
            print(f"\n  bootstrap {b+1}, {m}: {type(e).__name__}: {str(e)[:100]}")
    print(f"  bootstrap {b+1}/{N_BOOT}", end="\r", flush=True)
    del lg, ldf; gc.collect()

print("\n")
rows = []
for m, dims in boot.items():
    for dim, vals in dims.items():
        if len(vals) >= 5:
            v = np.asarray(vals, dtype=float)
            rows.append(dict(
                faculty=BOOT_FACULTY, role=BOOT_ROLE, notion=BOOT_NOTION,
                miner=m, dimension=dim, n_success=len(v),
                n_boot_requested=N_BOOT, traces_per_boot=min(BOOT_SAMPLE, len(eligible)),
                seed_base=BOOT_SEED, max_trace_len=BOOT_MAXLEN,
                mean=round(v.mean(), 4),
                median=round(np.median(v), 4),
                ci_lo=round(np.percentile(v, 2.5), 4),
                ci_hi=round(np.percentile(v, 97.5), 4)))
ci = pd.DataFrame(rows)
save(ci, "05_bootstrap_ci")

meta = {
    "python": sys.version,
    "pandas": pd.__version__,
    "pm4py": pm4py.__version__,
    "faculty": BOOT_FACULTY,
    "role": BOOT_ROLE,
    "case_notion": BOOT_NOTION,
    "n_boot": N_BOOT,
    "traces_per_boot": min(BOOT_SAMPLE, len(eligible)),
    "max_trace_len": BOOT_MAXLEN,
    "seed_base": BOOT_SEED,
    "resampling_unit": "trace/case",
    "replacement": True,
    "duplicate_draws_preserved": True,
    "miner_variants": {
        "alpha": "PM4Py Alpha default",
        "heuristic": "Variants.CLASSIC",
        "inductive": "Variants.IM"
    }
}
save(meta, "05_bootstrap_metadata")

for r in ci.itertuples():
    print(f"{r.miner:<10} {r.dimension:<14} {r.mean:.4f}  "
          f"95% CI [{r.ci_lo:.4f}, {r.ci_hi:.4f}]  "
          f"(successful={r.n_success}/{r.n_boot_requested})")
ci
